In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("/content/Titanic-Dataset.csv")

# ------------------ Leaky Pipeline ------------------

X = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])
y = df["Survived"]

num_cols = ["Age", "Fare", "SibSp", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

X[num_cols] = SimpleImputer(strategy="median").fit_transform(X[num_cols])
X[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(X[cat_cols])

X[num_cols] = StandardScaler().fit_transform(X[num_cols])

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(X[cat_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X.index,
)

X = pd.concat([X.drop(columns=cat_cols), encoded_df], axis=1)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(X_train_l, y_train_l)

leaky_pred = leaky_model.predict(X_test_l)
leaky_acc = accuracy_score(y_test_l, leaky_pred)

# ---------------- Correct Pipeline ----------------

df = pd.read_csv("/content/Titanic-Dataset.csv")

X = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

pipeline.fit(X_train, y_train)

correct_pred = pipeline.predict(X_test)
correct_acc = accuracy_score(y_test, correct_pred)

# ---------------- Results ----------------

results = pd.DataFrame(
    {
        "Pipeline": ["Leaky Pipeline", "Correct Pipeline"],
        "Test Accuracy": [leaky_acc, correct_acc],
    }
)

print(results)

           Pipeline  Test Accuracy
0    Leaky Pipeline       0.804469
1  Correct Pipeline       0.804469
